In [ ]:
# Import 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from textblob import TextBlob

from scipy.stats import (
    pearsonr,
    spearmanr,
    ttest_ind,
    chi2_contingency
)

import statsmodels.api as sm
import statsmodels.formula.api as smf

from IPython.display import display, Markdown
from ollama import chat

In [ ]:
#Ollama AI Summary 
def ai_summary(title, results_text):
    """
    Uses Ollama to interpret statistical results.
    """

    prompt = f"""
You are a data analyst explaining statistical results in simple terms.

Analyze the following output:

TITLE: {title}

RESULTS:
{results_text}

Rules:
- Explain what the results mean in plain English
- Do NOT repeat raw numbers
- Highlight key patterns or differences
- Mention if results suggest meaningful differences
- Keep it short (5–10 sentences max)
"""

    response = chat(
        model="llama3.2",
        messages=[{"role": "user", "content": prompt}]
    )

    return response["message"]["content"]




In [ ]:
input_file = 'reviews_BlackOps7_first1000.csv'
df = pd.read_csv(input_file)

print(f"Loaded {len(df)} rows")
display(df.head())

In [ ]:
#Sentiment and Categorization
def get_sentiment(text):
    """
    Returns sentiment polarity score (-1 to 1).
    """
    if pd.isna(text):
        return 0.0

    return TextBlob(str(text)).sentiment.polarity

In [ ]:

# --------------------------
# Categorization
# --------------------------
def categorize_review(review):

    usability_keywords = [
        "bug",
        "crash",
        "lag",
        "fps",
        "server",
        "performance",

        "glitch",
        "stutter",
        "freeze",
        "disconnect",
        "error",
        "broken",
        "unoptimized",
        "latency",
        "ping",
        "hitbox",
        "connection",
        "frame rate",
        "loading",
        "memory leak",
    ]

    mechanic_keywords = [
        "movement",
        "weapon",
        "perk",
        "battle pass",
        "matchmaking",

        "loadout",
        "killstreak",
        "scorestreak",
        "operator",
        "map",
        "game mode",
        "respawn",
        "spawn",
        "ability",
        "class",
        "gunplay",
        "ttk",
        "time to kill",
        "recoil",
        "aim assist",
        "multiplayer",
        "zombies",
        "campaign",
        "season",
    ]

    comparison_keywords = [
        "better than",
        "worse than",
        "compared to",
        "similar to",
        
        "just like",
        "same as",
        "feels like",
        "plays like",
        "reminds me of",
        "prefer",
        "used to",
        "previous",
        "original",
        "back in",
        "inferior to",
        "superior to",
        "rip off",
        "copy of",
        "old cod",
    ]

    # New categories based on look for categories AI analysis of full review file
    ai_content_keywords = [
        "slop",
        "ai generated",
        "ai slop",
        "no effort",
        "copy paste",
        "lazy",
        "generated content",
        "soulless",
        "cash grab",
    ]

    monetization_keywords = [
        "refund",
        "worth",
        "price",
        "sale",
        "buying",
        "paid",
        "microtransaction",
        "endgame",
        "season pass",
        "overpriced",
        "free to play",
        "cosmetic",
        "store",
        "bundle",
    ]

    story_keywords = [
        "campaign",
        "story",
        "single player",
        "solo",
        "narrative",
        "mission",
        "ending",
        "cutscene",
        "plot",
        "character",
    ]

    technical_requirements_keywords = [
        "secure boot",
        "tpm",
        "bios",
        "hardware",
        "anti-cheat",
        "ricochet",
        "kernel",
        "driver",
        "compatibility",
        "system requirement",
    ]

    sbmm_keywords = [
        "sbmm",
        "skill based",
        "lobbies",
        "lobby",
        "sweaty",
        "ranked",
        "casual",
        "matchmaking",
        "skill gap",
        "pub stomp",
    ]

    community_keywords = [
        "friends",
        "team",
        "squad",
        "banned",
        "toxic",
        "solo",
        "party",
        "coop",
        "co-op",
        "community",
        "chat",
        "voice",
        "grief",
    ]

    review_lower = review.lower()

    categories = []

    if any(word in review_lower for word in usability_keywords):
        categories.append("Usability Issues")

    if any(word in review_lower for word in mechanic_keywords):
        categories.append("New Mechanic Reception")

    if any(word in review_lower for word in comparison_keywords):
        categories.append("Competitive Analysis")

    if any(word in review_lower for word in ai_content_keywords):
        categories.append("AI-Generated Content Complaints")

    if any(word in review_lower for word in monetization_keywords):
        categories.append("Monetization & Value")

    if any(word in review_lower for word in story_keywords):
        categories.append("Story / Campaign")

    if any(word in review_lower for word in technical_requirements_keywords):
        categories.append("Technical Requirements")

    if any(word in review_lower for word in sbmm_keywords):
        categories.append("SBMM / Matchmaking Frustration")

    if any(word in review_lower for word in community_keywords):
        categories.append("Community & Social")

    if len(categories) == 0:
        categories.append("Game Reception")

    return categories

In [ ]:
sentiments = []
categories = []

for r in df["review"]:
    sentiments.append(get_sentiment(r))
    categories.append(", ".join(categorize_review(r)))

df["sentiment"] = sentiments
df["categories"] = categories

display(df.head())

In [ ]:
# Remove duplicate rows
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicates. {len(df)} rows remaining.")
display(df.head())

In [ ]:
#Feature engineering
df["review_length"] = df["review"].astype(str).str.len()
df["word_count"] = df["review"].astype(str).str.split().str.len()
df["sentiment_strength"] = df["sentiment"].abs()

print("Feature engineering complete.")
display(df[["review_length", "word_count", "sentiment", "sentiment_strength"]].head())

In [ ]:
# Category Columns
all_categories = [
    "Usability_Issues",
    "New_Mechanic_Reception",
    "Competitive_Analysis",
    "AI_Generated_Content_Complaints",
    "Monetization_and_Value",
    "Story___Campaign",
    "Technical_Requirements",
    "SBMM___Matchmaking_Frustration",
    "Community_and_Social",
    "Game_Reception"
]

for category in all_categories:
    df[category] = df["categories"].apply(lambda x: category in str(x))

print("Category columns created:")
print(df[all_categories].head())

In [ ]:
df[all_categories].sum().sort_values(ascending=False)

In [ ]:
%pip install textblob


In [ ]:
# AI Statistical Summary Function
# ------------------------------------

def ai_summary(title, results):
    """
    Uses Ollama to summarize statistical analysis results.
    """

    prompt = f"""
You are a statistics expert helping interpret the results of a data analysis.

Analysis:
{title}

Results:
{results}

Write a concise summary (2–4 sentences).

Your summary should:
- Explain the results in plain English.
- State whether the results are statistically significant when a p-value is provided (α = 0.05).
- Explain what the results mean.
- Do not speculate beyond the data.
"""

    response = chat(
        model="llama3.2",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.message.content


def display_ai_summary(title, results):

    summary = ai_summary(title, results)

    display(Markdown("### 🤖 AI Statistical Interpretation"))

    display(Markdown(summary))

In [ ]:
# Run sentiment analysis and categorization, then save to a new CSV
sentiments = []
categories = []

for r in df['review']:
    sentiments.append(get_sentiment(r))
    categories.append(', '.join(categorize_review(r)))

df['sentiment'] = sentiments
df['categories'] = categories

display(df)

In [ ]:
# Add a boolean column for each possible category
all_categories = [
    "Usability Issues",
    "New Mechanic Reception",
    "Competitive Analysis",
    "AI-Generated Content Complaints",
    "Monetization & Value",
    "Story___Campaign",
    "Technical Requirements",
    "SBMM / Matchmaking Frustration",
    "Community & Social",
    "Game Reception",
]

for category in all_categories:
    col_name = category.replace(" ", "_").replace("/", "_").replace("&", "and").replace("-", "_")
    col_values = []
    for cat_list in df['categories']:
        col_values.append(category in cat_list)
    df[col_name] = col_values

display(df.head(20))


In [ ]:
# Category Columns
# ------------------------------------

# Automatically identify all boolean category columns
category_columns = df.select_dtypes(include="bool").columns.tolist()

print(category_columns)

In [ ]:
all_categories = [
    "Usability Issues",
    "New Mechanic Reception",
    "Competitive Analysis",
    "AI-Generated Content Complaints",
    "Monetization & Value",
    "Story / Campaign",
    "Technical Requirements",
    "SBMM / Matchmaking Frustration",
    "Community & Social",
    "Game Reception"
]

for category in all_categories:
    df[category] = df["categories"].apply(
        lambda x: category in x if isinstance(x, str) else False
    )


In [ ]:
analyzed_file = input_file.replace('.csv', '_analyzed.csv')

df.to_csv(analyzed_file, index=False)
print(f"Saved {len(df)} analyzed reviews to {analyzed_file}")
df[['review', 'sentiment', 'categories']].head()


In [ ]:
# Read the analyzed CSV back and convert categories from string to list
df_analyzed = pd.read_csv(analyzed_file)

# Convert the categories column from comma-separated string to list
categories_list = []
for cat_string in df_analyzed['categories']:
    cat_list = []
    for cat in cat_string.split(','):
        cat_list.append(cat.strip())
    categories_list.append(cat_list)

df_analyzed['categories'] = categories_list

print(f"Loaded {len(df_analyzed)} analyzed reviews")
print("\nFirst few rows with categories as lists:")
display(df_analyzed)


In [ ]:
# Analysis suggestions:
# What is the number of each of the categories used? (what is the most/least common category)

# Exploratory Data Analysis 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from scipy.stats import (
    pearsonr,
    spearmanr,
    ttest_ind,
    chi2_contingency
)

import statsmodels.api as sm
import statsmodels.formula.api as smf

In [ ]:
# Feature Engineering
# -----------------------------------

df["review_length"] = df["review"].str.len()

df["word_count"] = df["review"].str.split().str.len()

df["sentiment_strength"] = df["sentiment"].abs()

display(df.head())

In [ ]:
#Category frequency analysis 

category_counts = (
    df[category_columns]
    .sum()
    .sort_values(ascending=False)
)

display(category_counts)

In [ ]:
# Analysis Functions (Visuals + Stats + AI Summary)
# ----------------------------
# 1. Histogram
# ----------------------------
def analyze_histogram(df, column, title):
    plt.figure(figsize=(8,5))
    df[column].hist(bins=30)
    plt.title(title)
    plt.xlabel(column)
    plt.ylabel("Frequency")
    plt.show()

    display_ai_summary(title, df[column].describe().to_string())


# ----------------------------
# 2. Bar Chart
# ----------------------------
def analyze_bar_chart(data, title):
    plt.figure(figsize=(10,6))
    data.plot(kind="bar")
    plt.title(title)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    display_ai_summary(title, data.to_string())


# ----------------------------
# 3. Box Plots
# ----------------------------
def analyze_boxplot(df, category_column, value_column, title):
    df.boxplot(column=value_column, by=category_column)
    plt.title(title)
    plt.suptitle("")
    plt.xticks(rotation=45)
    plt.show()

    results = (
        df.groupby(category_column)[value_column]
          .describe()
          .to_string()
    )

    display_ai_summary(title, results)

analyze_boxplot(
    df,
    "SBMM___Matchmaking_Frustration",
    "other_votes",
    "Helpful Votes for SBMM Reviews"
)

analyze_boxplot(
    df,
    "Usability_Issues",
    "other_votes",
    "Helpful Votes for Technical Issues"
)

analyze_boxplot(
    df,
    "Monetization_and_Value",
    "other_votes",
    "Helpful Votes for Monetization Reviews"
)
#Review length analysis boxplot
analyze_boxplot(
    df,
    "Story___Campaign",
    "review_length",
    "Review Length for Story Reviews"
)
analyze_boxplot(
    df,
    "Community_and_Social",
    "word_count",
    "Word Count for Community Reviews"
)
#Weighted Vote Score Boxplot
analyze_boxplot(df,"Game_Reception","weighted_vote_score","Weighted Vote Score for Game Reception Reviews"
)

# ----------------------------
# 4. Correlation Matrix
# ----------------------------
def analyze_correlation(df, columns, title):
    corr = df[columns].corr()

    display(corr)

    plt.figure(figsize=(8,6))
    plt.imshow(corr, cmap="coolwarm", interpolation="none")
    plt.colorbar()
    plt.xticks(range(len(columns)), columns, rotation=45, ha="right")
    plt.yticks(range(len(columns)), columns)
    plt.title(title)
    plt.tight_layout()
    plt.show()

    display_ai_summary(title, corr.to_string())


# ----------------------------
# 5. T-test helper
# ----------------------------
def analyze_ttest(df, group_col, value_col, title):
    group1 = df[df[group_col] == True][value_col]
    group2 = df[df[group_col] == False][value_col]

    stat, p = ttest_ind(group1, group2, nan_policy="omit")

    result_text = f"""
T-test Results:
Statistic: {stat:.4f}
P-value: {p:.6f}
Group 1 mean: {group1.mean():.4f}
Group 2 mean: {group2.mean():.4f}
"""

    print(result_text)
    display_ai_summary(title, result_text)


# ----------------------------
# 6. Chi-square test
# ----------------------------
def analyze_chi_square(df, col1, col2, title):
    contingency = pd.crosstab(df[col1], df[col2])
    chi2, p, dof, expected = chi2_contingency(contingency)

    result_text = f"""
Chi-square Results:
Chi2: {chi2:.4f}
P-value: {p:.6f}
Degrees of freedom: {dof}
"""

    print(result_text)
    display_ai_summary(title, result_text)
#7. Regression Analysis
def analyze_regression(df, x_columns, y_column, title):
    """
    Runs linear regression to test relationships between variables
    """

    # Drop missing values for clean model
    data = df[x_columns + [y_column]].dropna()

    X = data[x_columns]
    y = data[y_column]

    # Add intercept
    X = sm.add_constant(X)

    # Fit model
    model = sm.OLS(y, X).fit()

    # Print full statistical output
    print(model.summary())

    # AI interpretation
    display_ai_summary(title, model.summary().as_text())

In [ ]:
print("Overall Dataset Statistics")
display(df.describe())

print("\nSentiment Statistics")
display(df["sentiment"].describe())

print("\nReview Length Statistics")
display(df["review_length"].describe())

print("\nWord Count Statistics")
display(df["word_count"].describe())

print("Category Frequencies")

for category in all_categories:

    column = (
        category
        .replace(" ", "_")
        .replace("/", "_")
        .replace("&", "and")
        .replace("-", "_")
    )

    print(f"{category}: {df[column].sum()}")

In [ ]:
analyze_histogram(df, "sentiment", "Sentiment Distribution")

In [ ]:
#Exploratory Data Analysis
print("Dataset Shape")
print(df.shape)

print("\nColumns")
print(df.columns.tolist())

print("\nFirst Five Rows")
display(df.head())


In [ ]:
#Descriptive Statistics

print("Numeric Summary Statistics")

display(
    df[
        [
            "sentiment",
            "review_length",
            "word_count",
            "sentiment_strength"
        ]
    ].describe()
)

display_ai_summary(
    "Descriptive Statistics",
    df[
        [
            "sentiment",
            "review_length",
            "word_count",
            "sentiment_strength"
        ]
    ].describe().to_string()
)

In [ ]:
#Histograms
analyze_histogram(
    df,
    "sentiment",
    "Sentiment Distribution"
)

analyze_histogram(
    df,
    "review_length",
    "Review Length Distribution"
)

analyze_histogram(
    df,
    "word_count",
    "Word Count Distribution"
)

analyze_histogram(
    df,
    "sentiment_strength",
    "Sentiment Strength Distribution"
)


In [ ]:
#Boxplots for each category vs sentiment
analyze_boxplot(
    df,
    "Usability_Issues",
    "sentiment",
    "Usability Issues vs Sentiment"
)

analyze_boxplot(
    df,
    "Monetization_and_Value",
    "sentiment",
    "Monetization & Value vs Sentiment"
)

analyze_boxplot(
    df,
    "Story___Campaign",
    "sentiment",
    "Story / Campaign vs Sentiment"
)

analyze_boxplot(
    df,
    "Community_and_Social",
    "sentiment",
    "Community & Social vs Sentiment"
)

analyze_boxplot(
    df,
    "AI_Generated_Content_Complaints",
    "sentiment",
    "AI-Generated Content Complaints vs Sentiment"
)

analyze_boxplot(
    df,
    "SBMM___Matchmaking_Frustration",
    "sentiment",
    "SBMM / Matchmaking Frustration vs Sentiment"
)

In [ ]:
#Statistical Tests
analyze_ttest(
    df,
    "Usability_Issues",
    "sentiment",
    "T-test: Usability Issues vs Sentiment"
)


In [ ]:
analyze_chi_square(
    df,
    "Monetization_and_Value",
    "SBMM___Matchmaking_Frustration",
    "Chi-square: Monetization vs SBMM"
)


In [ ]:
analyze_regression(
    df,
    ["review_length", "word_count"],
    "sentiment",
    "Regression: Review Features Predicting Sentiment"
)

In [ ]:
# Executive Summary
def generate_executive_summary(df):

    print("\n" + "="*60)
    print("           STEAM REVIEW INSIGHTS SUMMARY")
    print("="*60)

    print(f"\nTotal Reviews Analyzed: {len(df)}")

    # Most discussed category
    category_counts = df[all_categories].sum().sort_values(ascending=False)

    print(f"\nMost Discussed Topic:")
    print(f"  {category_counts.idxmax()} ({category_counts.max()} reviews)")

    # Recommendation rate
    recommendation_rate = df["author_vote"].mean() * 100

    print(f"\nRecommendation Rate:")
    print(f"  {recommendation_rate:.1f}%")

    # Helpful votes
    print(f"\nAverage Helpful Votes:")
    print(f"  {df['other_votes'].mean():.2f}")

    # Review length
    print(f"\nAverage Review Length:")
    print(f"  {df['word_count'].mean():.1f} words")

    # Most helpful review
    top_review = df.loc[df["other_votes"].idxmax()]

    print("\nMost Helpful Review")
    print(f"Helpful Votes: {top_review['other_votes']}")
    print(f"Categories: {top_review['categories']}")
    print(f"Review:")
    print(top_review["review"][:300] + "...")

 #AI Summary of Executive Summary
print("\nAI OVERALL SUMMARY")

# Use the 25 most helpful reviews instead of a random sample
top_reviews = "\n".join(
    df.sort_values("other_votes", ascending=False)
      .head(25)["review"]
      .tolist()
)

summary = ai_summary(
    "Executive Summary of Player Feedback",
    top_reviews
)

print(summary)

In [ ]:
# ============================
# 8. FULL AUTOMATED PIPELINE
# ============================

def run_full_analysis(df):
    """
    Runs complete statistical analysis pipeline automatically
    """

    print("🚀 Starting Full Analysis Pipeline...\n")

    # -----------------------
    # 1. CATEGORY FREQUENCY
    # -----------------------
    category_counts = df[all_categories].sum().sort_values(ascending=False)

    analyze_bar_chart(category_counts, "Category Frequency Distribution")

    # -----------------------
    # 2. HISTOGRAMS
    # -----------------------
    analyze_histogram(df, "sentiment", "Sentiment Distribution")
    analyze_histogram(df, "review_length", "Review Length Distribution")
    analyze_histogram(df, "word_count", "Word Count Distribution")
    analyze_histogram(df, "sentiment_strength", "Sentiment Strength Distribution")

    # -----------------------
    # 3. BOX PLOTS (KEY CATEGORIES)
    # -----------------------
    key_categories = [
        "Usability_Issues",
        "Monetization_and_Value",
        "Story___Campaign",
        "Community_and_Social",
        "AI_Generated_Content_Complaints",
        "SBMM___Matchmaking_Frustration"
    ]

    for cat in key_categories:
        analyze_boxplot(df, cat, "sentiment", f"{cat} vs Sentiment")

    # -----------------------
    # 4. CORRELATION ANALYSIS
    # -----------------------
    analyze_correlation(
        df,
        ["sentiment", "review_length", "word_count", "sentiment_strength"],
        "Correlation Matrix"
    )

    # -----------------------
    # 5. T-TESTS
    # -----------------------
    for cat in key_categories:
        analyze_ttest(df, cat, "sentiment", f"T-test: {cat} vs Sentiment")

    # -----------------------
    # 6. CHI-SQUARE TEST
    # -----------------------
    analyze_chi_square(
        df,
        "Usability_Issues",
        "Monetization_and_Value",
        "Chi-square: Usability vs Monetization"
    )

    # -----------------------
    # 7. REGRESSION
    # -----------------------
    analyze_regression(
        df,
        ["review_length", "word_count"],
        "sentiment",
        "Regression: Review Features Predicting Sentiment"
    )
    #----------------------
    # EXECUTIVE SUMMARY
    # -----------------------
    generate_executive_summary(df)
    print("\n✅ Full Analysis Complete!")


In [ ]:
run_full_analysis(df)